In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

AS_OF_DATE = pd.Timestamp("2026-09-22")

BASE_DIR = Path(
    r"C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned"
)

OUTPUT_DIR = BASE_DIR / "finance"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

rng = np.random.default_rng(42)

In [3]:
journeys = pd.read_csv(
    BASE_DIR / "student_journeys_clean.csv",
    dtype={
        "sj_id": "string",
        "s_id": "string",
        "c_id": "string",
        "mc_id": "string"
    }
)

students = pd.read_csv(
    BASE_DIR / "students_clean.csv",
    dtype={
        "s_id": "string"
    }
)

applications = pd.read_csv(
    BASE_DIR / "university_applications_clean.csv",
    dtype={
        "app_id": "string",
        "sj_id": "string"
    }
)

visas = pd.read_csv(
    BASE_DIR / "visa_applications_clean.csv",
    dtype={
        "visa_id": "string",
        "sj_id": "string",
        "app_id": "string"
    }
)

In [5]:
journeys["lead_date"] = pd.to_datetime(
    journeys["lead_date"],
    errors="coerce"
)

In [7]:
print("Student journeys:", len(journeys))
print("University applications:", len(applications))
print("Visa applications:", len(visas))

print("\nVisa statuses:")
print(
    visas["visa_status"]
    .value_counts()
)

print("\nApproved visa journey stages:")

approved_stage_check = (
    visas[
        visas["visa_status"] == "Approved"
    ]
    .merge(
        journeys[
            ["sj_id", "current_stage"]
        ],
        on="sj_id",
        how="left"
    )
)

print(
    approved_stage_check[
        "current_stage"
    ].value_counts()
)

Student journeys: 1000
University applications: 1118
Visa applications: 289

Visa statuses:
visa_status
Approved            135
Pending Decision     80
Preparing            74
Name: count, dtype: int64

Approved visa journey stages:
current_stage
Visa Approved         75
Arrived in Ireland    60
Name: count, dtype: int64


In [9]:
partners = pd.DataFrame([
    {
        "university_name": "Trinity College Dublin",
        "rank_band": "Top 150",
        "partnership_tier": "Platinum",
        "tuition_min_eur": 22000,
        "tuition_max_eur": 30000,
        "commission_rate": 0.18
    },
    {
        "university_name": "University College Dublin",
        "rank_band": "Top 150",
        "partnership_tier": "Platinum",
        "tuition_min_eur": 20000,
        "tuition_max_eur": 28000,
        "commission_rate": 0.18
    },
    {
        "university_name": "University College Cork",
        "rank_band": "151-300",
        "partnership_tier": "Gold",
        "tuition_min_eur": 18000,
        "tuition_max_eur": 24000,
        "commission_rate": 0.165
    },
    {
        "university_name": "University of Galway",
        "rank_band": "151-300",
        "partnership_tier": "Gold",
        "tuition_min_eur": 17000,
        "tuition_max_eur": 23000,
        "commission_rate": 0.16
    },
    {
        "university_name": "University of Limerick",
        "rank_band": "301-500",
        "partnership_tier": "Gold",
        "tuition_min_eur": 16500,
        "tuition_max_eur": 22000,
        "commission_rate": 0.155
    },
    {
        "university_name": "Dublin City University",
        "rank_band": "301-500",
        "partnership_tier": "Silver",
        "tuition_min_eur": 15500,
        "tuition_max_eur": 21000,
        "commission_rate": 0.15
    },
    {
        "university_name": "Maynooth University",
        "rank_band": "501-800",
        "partnership_tier": "Silver",
        "tuition_min_eur": 15000,
        "tuition_max_eur": 20000,
        "commission_rate": 0.135
    },
    {
        "university_name": "Technological University Dublin",
        "rank_band": "501-800",
        "partnership_tier": "Silver",
        "tuition_min_eur": 13500,
        "tuition_max_eur": 18000,
        "commission_rate": 0.13
    },
    {
        "university_name": "National College of Ireland",
        "rank_band": "Partner Institution",
        "partnership_tier": "Bronze",
        "tuition_min_eur": 13000,
        "tuition_max_eur": 17000,
        "commission_rate": 0.12
    },
    {
        "university_name": "Griffith College",
        "rank_band": "Partner Institution",
        "partnership_tier": "Bronze",
        "tuition_min_eur": 12500,
        "tuition_max_eur": 16500,
        "commission_rate": 0.115
    },
    {
        "university_name": "Dublin Business School",
        "rank_band": "Partner Institution",
        "partnership_tier": "Bronze",
        "tuition_min_eur": 12000,
        "tuition_max_eur": 16000,
        "commission_rate": 0.11
    }
])

In [11]:
missing_universities = sorted(
    set(
        applications[
            "university_name"
        ].dropna().unique()
    )
    -
    set(
        partners[
            "university_name"
        ].unique()
    )
)

print(
    "Universities missing commercial rules:",
    missing_universities
)

Universities missing commercial rules: []


In [13]:
finance = (
    visas[
        visas["visa_status"] == "Approved"
    ]
    .merge(
        applications,
        on=[
            "app_id",
            "sj_id"
        ],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        journeys[
            [
                "sj_id",
                "s_id",
                "c_id",
                "lead_date",
                "current_stage",
                "case_status",
                "target_intake_term",
                "target_intake_year"
            ]
        ],
        on="sj_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        partners,
        on="university_name",
        how="left",
        validate="many_to_one"
    )
)

In [15]:
print(
    "Finance candidates:",
    len(finance)
)

Finance candidates: 135


In [17]:
assert finance["sj_id"].notna().all()
assert finance["app_id"].notna().all()
assert finance["visa_id"].notna().all()

assert (
    finance["visa_status"] == "Approved"
).all()

assert (
    finance["application_status"]
    == "Offer Issued"
).all()

assert (
    finance["application_outcome"]
    == "Offer"
).all()

assert (
    finance["current_stage"]
    .isin([
        "Visa Approved",
        "Arrived in Ireland"
    ])
).all()

assert finance[
    "commission_rate"
].notna().all()

print(
    "SOURCE COMPATIBILITY CHECKS PASSED"
)

SOURCE COMPATIBILITY CHECKS PASSED


In [19]:
finance["revenue_type"] = np.select(
    [
        finance[
            "current_stage"
        ].eq(
            "Arrived in Ireland"
        ),

        finance[
            "current_stage"
        ].eq(
            "Visa Approved"
        )
    ],
    [
        "Earned",
        "Forecast"
    ],
    default="Review"
)

In [21]:
print(
    finance[
        "revenue_type"
    ].value_counts()
)

revenue_type
Forecast    75
Earned      60
Name: count, dtype: int64


In [23]:
assert (
    finance[
        "revenue_type"
    ] != "Review"
).all()

In [25]:
finance[
    "estimated_tuition_eur"
] = [
    rng.uniform(
        low,
        high
    )
    for low, high in zip(
        finance[
            "tuition_min_eur"
        ],
        finance[
            "tuition_max_eur"
        ]
    )
]

In [27]:
finance[
    "estimated_tuition_eur"
] = (
    finance[
        "estimated_tuition_eur"
    ]
    .div(100)
    .round()
    .mul(100)
)

In [29]:
finance[
    "commission_amount_eur"
] = (
    finance[
        "estimated_tuition_eur"
    ]
    *
    finance[
        "commission_rate"
    ]
).round(2)

In [31]:
expected_commission = (
    finance[
        "estimated_tuition_eur"
    ]
    *
    finance[
        "commission_rate"
    ]
).round(2)

assert (
    finance[
        "commission_amount_eur"
    ]
    ==
    expected_commission
).all()

print(
    "Commission calculations valid"
)

Commission calculations valid


In [33]:
finance = finance.reset_index(
    drop=True
)

finance[
    "commission_id"
] = [
    f"COM{i:04d}"
    for i in range(
        1,
        len(finance) + 1
    )
]

In [35]:
finance[
    "earned_date"
] = pd.NaT

finance[
    "invoice_date"
] = pd.NaT

finance[
    "due_date"
] = pd.NaT

finance[
    "paid_date"
] = pd.NaT

finance[
    "commission_status"
] = "Forecast"

In [37]:
earned_indices = finance.index[
    finance[
        "revenue_type"
    ] == "Earned"
]

for idx in earned_indices:

    lead_date = finance.at[
        idx,
        "lead_date"
    ]

    earliest = (
        lead_date
        +
        pd.Timedelta(
            days=120
        )
    )

    latest = (
        AS_OF_DATE
        -
        pd.Timedelta(
            days=2
        )
    )

    if earliest > latest:
        earliest = (
            lead_date
            +
            pd.Timedelta(
                days=30
            )
        )

    available_days = (
        latest - earliest
    ).days

    if available_days < 0:
        earned_date = latest
    else:
        earned_date = (
            earliest
            +
            pd.Timedelta(
                days=int(
                    rng.integers(
                        0,
                        available_days + 1
                    )
                )
            )
        )

    finance.at[
        idx,
        "earned_date"
    ] = earned_date

In [39]:
for idx in earned_indices:

    earned_date = finance.at[
        idx,
        "earned_date"
    ]

    invoice_delay = int(
        rng.integers(
            3,
            11
        )
    )

    proposed_invoice = (
        earned_date
        +
        pd.Timedelta(
            days=invoice_delay
        )
    )

    if proposed_invoice <= AS_OF_DATE:

        finance.at[
            idx,
            "invoice_date"
        ] = proposed_invoice

        finance.at[
            idx,
            "due_date"
        ] = (
            proposed_invoice
            +
            pd.Timedelta(
                days=30
            )
        )

        finance.at[
            idx,
            "commission_status"
        ] = "Invoiced"

    else:

        finance.at[
            idx,
            "commission_status"
        ] = "Earned - Not Invoiced"

In [41]:
invoiced_indices = finance.index[
    finance[
        "invoice_date"
    ].notna()
]

for idx in invoiced_indices:

    invoice_date = finance.at[
        idx,
        "invoice_date"
    ]

    due_date = finance.at[
        idx,
        "due_date"
    ]

    days_since_invoice = (
        AS_OF_DATE
        -
        invoice_date
    ).days

    # CASE 1:
    # Invoice is already overdue
    if due_date < AS_OF_DATE:

        if rng.random() < 0.80:

            payment_window = (
                AS_OF_DATE
                -
                invoice_date
            ).days

            delay = int(
                rng.integers(
                    1,
                    payment_window + 1
                )
            )

            paid_date = (
                invoice_date
                +
                pd.Timedelta(
                    days=delay
                )
            )

            finance.at[
                idx,
                "paid_date"
            ] = paid_date

            finance.at[
                idx,
                "commission_status"
            ] = "Paid"

        else:

            finance.at[
                idx,
                "commission_status"
            ] = "Overdue"

    # CASE 2:
    # Invoice is not yet due
    else:

        # Some universities pay before due date
        if (
            days_since_invoice >= 5
            and
            rng.random() < 0.40
        ):

            delay = int(
                rng.integers(
                    1,
                    days_since_invoice + 1
                )
            )

            paid_date = (
                invoice_date
                +
                pd.Timedelta(
                    days=delay
                )
            )

            finance.at[
                idx,
                "paid_date"
            ] = paid_date

            finance.at[
                idx,
                "commission_status"
            ] = "Paid"

        else:

            finance.at[
                idx,
                "commission_status"
            ] = "Invoiced"

In [43]:
finance_final = finance[
    [
        "commission_id",
        "sj_id",
        "s_id",
        "app_id",
        "visa_id",
        "university_name",
        "partnership_tier",
        "rank_band",
        "estimated_tuition_eur",
        "commission_rate",
        "commission_amount_eur",
        "revenue_type",
        "commission_status",
        "earned_date",
        "invoice_date",
        "due_date",
        "paid_date"
    ]
].copy()

In [45]:
print(
    "Future earned dates:",
    (
        finance_final[
            "earned_date"
        ]
        > AS_OF_DATE
    ).sum()
)

print(
    "Future invoice dates:",
    (
        finance_final[
            "invoice_date"
        ]
        > AS_OF_DATE
    ).sum()
)

print(
    "Future paid dates:",
    (
        finance_final[
            "paid_date"
        ]
        > AS_OF_DATE
    ).sum()
)

Future earned dates: 0
Future invoice dates: 0
Future paid dates: 0


In [47]:
print(
    "Invoice before earned:",
    (
        finance_final[
            "invoice_date"
        ].notna()
        &
        (
            finance_final[
                "invoice_date"
            ]
            <
            finance_final[
                "earned_date"
            ]
        )
    ).sum()
)

print(
    "Due before invoice:",
    (
        finance_final[
            "due_date"
        ].notna()
        &
        (
            finance_final[
                "due_date"
            ]
            <
            finance_final[
                "invoice_date"
            ]
        )
    ).sum()
)

print(
    "Paid before invoice:",
    (
        finance_final[
            "paid_date"
        ].notna()
        &
        (
            finance_final[
                "paid_date"
            ]
            <
            finance_final[
                "invoice_date"
            ]
        )
    ).sum()
)

Invoice before earned: 0
Due before invoice: 0
Paid before invoice: 0


In [49]:
forecast_rows = finance_final[
    finance_final[
        "revenue_type"
    ] == "Forecast"
]

forecast_with_dates = (
    forecast_rows[
        [
            "earned_date",
            "invoice_date",
            "due_date",
            "paid_date"
        ]
    ]
    .notna()
    .any(axis=1)
    .sum()
)

print(
    "Forecast rows with dates:",
    forecast_with_dates
)

Forecast rows with dates: 0


In [51]:
paid_rows = finance_final[
    finance_final[
        "commission_status"
    ] == "Paid"
]

print(
    "Paid without paid_date:",
    paid_rows[
        "paid_date"
    ].isna().sum()
)

Paid without paid_date: 0


In [53]:
print(
    "Payments after snapshot:",
    (
        paid_rows[
            "paid_date"
        ]
        > AS_OF_DATE
    ).sum()
)

Payments after snapshot: 0


In [55]:
overdue_rows = finance_final[
    finance_final[
        "commission_status"
    ] == "Overdue"
]

invalid_overdue = overdue_rows[
    ~(
        overdue_rows[
            "due_date"
        ]
        < AS_OF_DATE
    )
]

print(
    "Invalid overdue rows:",
    len(
        invalid_overdue
    )
)

Invalid overdue rows: 0


In [57]:
visa_check = (
    finance_final[
        [
            "visa_id",
            "sj_id",
            "app_id"
        ]
    ]
    .merge(
        visas[
            [
                "visa_id",
                "sj_id",
                "app_id",
                "visa_status"
            ]
        ],
        on=[
            "visa_id",
            "sj_id",
            "app_id"
        ],
        how="left",
        indicator=True
    )
)

In [59]:
print(
    "Broken finance → visa links:",
    (
        visa_check[
            "_merge"
        ] != "both"
    ).sum()
)

print(
    "Finance records with non-approved visa:",
    (
        visa_check[
            "visa_status"
        ] != "Approved"
    ).sum()
)

Broken finance → visa links: 0
Finance records with non-approved visa: 0


In [61]:
app_check = (
    finance_final[
        [
            "app_id",
            "sj_id",
            "university_name"
        ]
    ]
    .merge(
        applications[
            [
                "app_id",
                "sj_id",
                "university_name",
                "application_status",
                "application_outcome"
            ]
        ],
        on=[
            "app_id",
            "sj_id",
            "university_name"
        ],
        how="left",
        indicator=True
    )
)

In [63]:
print(
    "Broken finance → application links:",
    (
        app_check[
            "_merge"
        ] != "both"
    ).sum()
)

print(
    "Finance records without Offer Issued:",
    (
        app_check[
            "application_status"
        ] != "Offer Issued"
    ).sum()
)

print(
    "Finance records without Offer outcome:",
    (
        app_check[
            "application_outcome"
        ] != "Offer"
    ).sum()
)

Broken finance → application links: 0
Finance records without Offer Issued: 0
Finance records without Offer outcome: 0


In [65]:
journey_check = (
    finance_final[
        [
            "sj_id",
            "s_id",
            "revenue_type"
        ]
    ]
    .merge(
        journeys[
            [
                "sj_id",
                "s_id",
                "current_stage"
            ]
        ],
        on=[
            "sj_id",
            "s_id"
        ],
        how="left",
        indicator=True
    )
)

In [67]:
print(
    "Broken finance → journey links:",
    (
        journey_check[
            "_merge"
        ] != "both"
    ).sum()
)

Broken finance → journey links: 0


In [69]:
earned_invalid = journey_check[
    journey_check[
        "revenue_type"
    ].eq("Earned")
    &
    ~journey_check[
        "current_stage"
    ].eq("Arrived in Ireland")
]

forecast_invalid = journey_check[
    journey_check[
        "revenue_type"
    ].eq("Forecast")
    &
    ~journey_check[
        "current_stage"
    ].eq("Visa Approved")
]

print(
    "Earned commissions with wrong journey stage:",
    len(earned_invalid)
)

print(
    "Forecast commissions with wrong journey stage:",
    len(forecast_invalid)
)

Earned commissions with wrong journey stage: 0
Forecast commissions with wrong journey stage: 0


In [71]:
print(
    "\nRevenue Type:"
)

print(
    finance_final[
        "revenue_type"
    ].value_counts()
)

print(
    "\nCommission Status:"
)

print(
    finance_final[
        "commission_status"
    ].value_counts()
)

print(
    "\nTotal Finance Records:",
    len(finance_final)
)


Revenue Type:
revenue_type
Forecast    75
Earned      60
Name: count, dtype: int64

Commission Status:
commission_status
Forecast                 75
Paid                     31
Invoiced                 16
Overdue                   8
Earned - Not Invoiced     5
Name: count, dtype: int64

Total Finance Records: 135


In [73]:
display(
    finance_final[
        finance_final[
            "revenue_type"
        ] == "Earned"
    ].head(20)
)

,commission_id,sj_id,s_id,app_id,visa_id,university_name,partnership_tier,rank_band,estimated_tuition_eur,commission_rate,commission_amount_eur,revenue_type,commission_status,earned_date,invoice_date,due_date,paid_date
0,COM0001,4,4,2,2,Dublin City University,Silver,301-500,19800.0,0.150,2970.0,Earned,Paid,2026-08-10,2026-08-17,2026-09-16,2026-08-31
4,COM0005,26,26,23,12,Trinity College Dublin,Platinum,Top 150,22800.0,0.180,4104.0,Earned,Overdue,2026-08-04,2026-08-09,2026-09-08,NaT
7,COM0008,65,65,67,20,Maynooth University,Silver,501-800,18900.0,0.135,2551.5,Earned,Paid,2026-08-28,2026-09-02,2026-10-02,2026-09-14
9,COM0010,81,81,85,23,University of Galway,Gold,151-300,19700.0,0.160,3152.0,Earned,Earned - Not Invoiced,2026-09-19,NaT,NaT,NaT
10,COM0011,87,87,93,24,Maynooth University,Silver,501-800,16900.0,0.135,2281.5,Earned,Paid,2026-08-22,2026-08-28,2026-09-27,2026-09-22
11,COM0012,88,88,95,25,Griffith College,Bronze,Partner Institution,16200.0,0.115,1863.0,Earned,Overdue,2026-05-13,2026-05-22,2026-06-21,NaT
15,COM0016,111,111,119,33,University of Limerick,Gold,301-500,17700.0,0.155,2743.5,Earned,Invoiced,2026-08-28,2026-09-07,2026-10-07,NaT
16,COM0017,119,119,126,35,University of Galway,Gold,151-300,20300.0,0.160,3248.0,Earned,Paid,2026-07-01,2026-07-06,2026-08-05,2026-09-05
17,COM0018,127,127,135,40,University College Cork,Gold,151-300,18400.0,0.165,3036.0,Earned,Paid,2026-08-23,2026-08-27,2026-09-26,2026-09-01
19,COM0020,147,147,164,45,University of Limerick,Gold,301-500,20000.0,0.155,3100.0,Earned,Paid,2026-08-29,2026-09-08,2026-10-08,2026-09-12


In [75]:
display(
    finance_final[
        finance_final[
            "revenue_type"
        ] == "Forecast"
    ].head(10)
)

,commission_id,sj_id,s_id,app_id,visa_id,university_name,partnership_tier,rank_band,estimated_tuition_eur,commission_rate,commission_amount_eur,revenue_type,commission_status,earned_date,invoice_date,due_date,paid_date
1,COM0002,9,9,6,4,University College Cork,Gold,151-300,20600.0,0.165,3399.0,Forecast,Forecast,NaT,NaT,NaT,NaT
2,COM0003,20,20,18,9,Technological University Dublin,Silver,501-800,17400.0,0.130,2262.0,Forecast,Forecast,NaT,NaT,NaT,NaT
3,COM0004,25,25,22,11,National College of Ireland,Bronze,Partner Institution,15800.0,0.120,1896.0,Forecast,Forecast,NaT,NaT,NaT,NaT
5,COM0006,34,34,32,13,Technological University Dublin,Silver,501-800,17900.0,0.130,2327.0,Forecast,Forecast,NaT,NaT,NaT,NaT
6,COM0007,61,61,63,18,Trinity College Dublin,Platinum,Top 150,28100.0,0.180,5058.0,Forecast,Forecast,NaT,NaT,NaT,NaT
8,COM0009,68,68,69,21,Dublin Business School,Bronze,Partner Institution,12500.0,0.110,1375.0,Forecast,Forecast,NaT,NaT,NaT,NaT
12,COM0013,97,97,104,28,Griffith College,Bronze,Partner Institution,15100.0,0.115,1736.5,Forecast,Forecast,NaT,NaT,NaT,NaT
13,COM0014,100,100,106,30,Maynooth University,Silver,501-800,19100.0,0.135,2578.5,Forecast,Forecast,NaT,NaT,NaT,NaT
14,COM0015,105,105,109,31,Dublin Business School,Bronze,Partner Institution,13800.0,0.110,1518.0,Forecast,Forecast,NaT,NaT,NaT,NaT
18,COM0019,136,136,149,43,Griffith College,Bronze,Partner Institution,15800.0,0.115,1817.0,Forecast,Forecast,NaT,NaT,NaT,NaT


In [77]:
assert len(finance_final) == 135

assert (
    finance_final[
        "commission_id"
    ].duplicated().sum()
    == 0
)

assert (
    finance_final[
        "visa_id"
    ].duplicated().sum()
    == 0
)

assert (
    finance_final[
        "app_id"
    ].duplicated().sum()
    == 0
)

assert (
    finance_final[
        "sj_id"
    ].duplicated().sum()
    == 0
)

assert (
    finance_final[
        "revenue_type"
    ].eq(
        "Earned"
    ).sum()
    == 60
)

assert (
    finance_final[
        "revenue_type"
    ].eq(
        "Forecast"
    ).sum()
    == 75
)

print(
    "✓ FINANCE DATASET PASSED ALL FINAL CHECKS"
)

✓ FINANCE DATASET PASSED ALL FINAL CHECKS


In [79]:
output_path = (
    OUTPUT_DIR
    /
    "university_commissions_clean.csv"
)

finance_final.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print(
    "Saved:",
    output_path
)

Saved: C:\Users\kmpsh\Downloads\Msc\Projects\Edu_Agency\datasets\cleaned\finance\university_commissions_clean.csv
